# 09 - Histogram processing

**Syllabus:** histogram processing (part of image enhancement in the spatial domain)

A histogram throws away *all* spatial information and keeps only "how many pixels of each
value". That sounds like a crippling loss, and it is exactly why it is useful: the resulting
statistic is tiny, cheap, and enough to drive automatic exposure, automatic thresholding,
matching one image's look to another's, and retrieval-by-appearance.

Everything in this notebook is a **LUT** built from the histogram, so it is all point processing —
but the LUT is now *derived from the image* instead of chosen by you.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
# the graders are the test harness -- everything else is written out in full
from cvlab import check_equalize, check_histogram, check_match_histogram

dark = cv2.imread("images/board.jpg", cv2.IMREAD_GRAYSCALE)
lena = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)

## Task 1 - the histogram

$$h(r_k) = n_k \qquad k = 0 \ldots 255$$

Just a tally. The **normalised** histogram `p(r_k) = n_k / MN` is an estimate of the probability
that a randomly chosen pixel has value `r_k` — which is what lets you treat the rest of this
notebook as probability theory.

In [ ]:
def my_calcHist(gray):
    """Count pixels at each of the 256 intensity levels.

    Parameters
    ----------
    gray : (H, W) uint8 array

    Returns
    -------
    length-256 integer array. Entry k is the number of pixels equal to k.
    The counts sum to gray.size.
    """
    # TODO: one call does it
    raise NotImplementedError

In [ ]:
check_histogram(my_calcHist)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.calcHist(images, channels, mask, histSize, ranges)
#   EVERY argument is a list, even for one image and one channel. This trips up everyone:
#   cv2.calcHist([img], [0], None, [256], [0, 256])
import cv2, numpy as np
cv_h = cv2.calcHist([dark], [0], None, [256], [0, 256]).ravel()
assert np.array_equal(cv_h.astype(np.int64), my_calcHist(dark).astype(np.int64))
print("match. Colour histogram, one call per channel:")
for i, ch in enumerate("RGB"):
    h = cv2.calcHist([cv2.cvtColor(cv2.imread("images/fruits.jpg"), cv2.COLOR_BGR2RGB)], [i], None, [256], [0, 256]).ravel()
    print(f"  {ch}: peak at intensity {int(h.argmax()):3d} with {int(h.max()):6d} pixels")
print("\n`mask` lets you histogram only part of an image -- pass a uint8 0/255 array.")
print("A 3-D colour histogram: cv2.calcHist([img], [0,1,2], None, [8,8,8], [0,256]*3)")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

`np.bincount(gray.ravel(), minlength=256)`. The `minlength` matters — without it an image
whose brightest pixel is 200 gives you a 201-long array and every later index breaks.

Doing it the explicit way (`for v in gray.ravel(): h[v] += 1`) is worth writing once to see
what `bincount` is doing, then throw it away.

</details>

In [ ]:
h = my_calcHist(dark)
cdf = h.cumsum() / h.sum()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].imshow(dark, cmap="gray", vmin=0, vmax=255); ax[0].set_title("board.jpg"); ax[0].axis("off")
ax[1].bar(np.arange(256), h, width=1); ax[1].set_title("my_calcHist h(r)"); ax[1].set_xlim(0,255)
ax[2].plot(cdf); ax[2].set_title("normalised CDF"); ax[2].set_xlim(0,255); ax[2].set_ylim(0,1)
ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"mean {dark.mean():.1f}  std {dark.std():.1f}  "
      f"used levels {(h>0).sum()}/256  99% of pixels below {np.searchsorted(cdf, 0.99)}")

Read that histogram: everything is jammed into the low third of the range, and more than half
the available levels are unused. That is a **low-contrast, dark** image, diagnosed without
looking at it — which is what an auto-exposure algorithm does.

## Histogram equalisation

The idea: find the transformation `s = T(r)` that makes the output histogram **flat**, so every
level is used equally often and no range of the display is wasted.

The derivation is short and worth knowing. Treat intensity as a random variable. If

$$s = T(r) = (L-1)\int_0^r p_r(w)\,dw$$

then it is a standard result that `s` is uniformly distributed. That integral is the CDF. So:

**the transformation is the cumulative histogram, scaled.**

In the discrete case:

$$s_k = \text{round}\left[(L-1)\sum_{j=0}^{k} p_r(r_j)\right]$$

### Task 2 - equalisation

The grader requires an **exact** match with `cv2.equalizeHist`, so use the exact form OpenCV
uses — which is also the textbook-correct one, because it guarantees the darkest pixel lands on 0:

$$s_k = \text{round}\left[\frac{\text{cdf}(k) - \text{cdf}(k_{\min})}{MN - h(k_{\min})} \times 255\right]$$

where `k_min` is the **lowest intensity that actually occurs**. Guard the case where every pixel
has the same value — the denominator is 0 there.

In [ ]:
def my_equalizeHist(gray):
    """Histogram equalisation, matching cv2.equalizeHist exactly.

    Build a 256-entry LUT from the cumulative my_calcHist:
        lut[k] = round((cdf[k] - cdf[kmin]) / (N - h[kmin]) * 255)
    where kmin is the lowest intensity present and N is the pixel count.

    Parameters
    ----------
    gray : (H, W) uint8 array

    Returns
    -------
    (H, W) uint8 array. min becomes 0, max becomes 255. A constant image is returned unchanged.
    """
    # TODO: my_calcHist -> cumsum -> find kmin -> build the LUT -> index with it
    raise NotImplementedError

In [ ]:
check_equalize(my_equalizeHist)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.equalizeHist(src) -- grayscale uint8 only, no options at all.
import cv2, numpy as np
assert np.array_equal(cv2.equalizeHist(dark), my_equalizeHist(dark)), "disagrees with cv2"
print("exact match.")
print("\nFor colour, equalise a BRIGHTNESS channel, never R/G/B separately:")
print("  ycc = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)")
print("  ycc[...,0] = cv2.equalizeHist(ycc[...,0])")
print("  out = cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)")
print("\nAdaptive version: clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))")
print("                  clahe.apply(gray)")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

`h = histogram(gray).astype(np.float64)` then `cdf = h.cumsum()`.

`kmin = np.nonzero(h)[0][0]` is the first non-empty bin.
`denom = gray.size - h[kmin]` — if that is `<= 0` every pixel is identical, so return a copy.

Applying a LUT is just fancy indexing: `lut[gray]`. That single expression replaces every
pixel with its table entry, and it is why point processing is so fast.

</details>

In [ ]:
eq = my_equalizeHist(dark)
fig, ax = plt.subplots(2, 3, figsize=(13, 7))
for row, (im, name) in enumerate([(dark, "before"), (eq, "after equalisation")]):
    hh = my_calcHist(im)
    ax[row,0].imshow(im, cmap="gray", vmin=0, vmax=255); ax[row,0].set_title(name); ax[row,0].axis("off")
    ax[row,1].bar(np.arange(256), hh, width=1); ax[row,1].set_xlim(0,255); ax[row,1].set_title("my_calcHist")
    ax[row,2].plot(hh.cumsum()/im.size); ax[row,2].set_xlim(0,255); ax[row,2].set_ylim(0,1)
    ax[row,2].set_title("CDF"); ax[row,2].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"std {dark.std():.1f} -> {eq.std():.1f}   levels used {(my_calcHist(dark)>0).sum()} -> {(my_calcHist(eq)>0).sum()}")

Notice the output histogram is **not** flat — it is *spread*, with gaps. It cannot be flat: a LUT
can merge levels but never split one, so a tall spike stays a tall spike and just moves. The CDF,
however, is now close to the straight diagonal, which is the honest statement of what
equalisation achieves.

That is also why equalisation sometimes looks awful: it happily amplifies noise in a large flat
region, because that region owns a big chunk of the histogram and therefore gets a big slice of
the output range.

In [ ]:
import cv2
_imgs, _titles = list([my_equalizeHist(cv2.imread(f"images/{n}", cv2.IMREAD_GRAYSCALE)) for n in ["board.jpg","lena.jpg","baboon.jpg","sudoku.png"]]), ["board (helps a lot)","lena (already fine)","baboon","sudoku (noise amplified)"]
fig, axes = plt.subplots(1, len(_imgs), figsize=(3.2 * len(_imgs), 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

## Histogram specification (matching)

Equalisation gives you no control — the target is always uniform. **Specification** lets you
demand an arbitrary target shape. The trick is to route through the uniform distribution:

- `s = T(r)` equalises the source
- `s = G(z)` equalises the reference
- therefore `z = G⁻¹(T(r))` maps source values to reference values

So: for each source level, find the reference level whose CDF value is closest. That is a
search in a sorted array, which is `np.searchsorted`.

### Task 3 - matching

In [ ]:
def my_match_histogram(src, ref):
    """Remap src so its my_calcHist approximates ref's. (Histogram specification.)

    For each source level k, find the smallest reference level z whose CDF value
    reaches the source CDF value at k. That mapping is the LUT.

    Parameters
    ----------
    src : (H, W) uint8 array, the image to transform
    ref : (H, W) uint8 array, whose my_calcHist is the target. Need not be the same size.

    Returns
    -------
    (H, W) uint8 array, same shape as src. Monotonic in the input.
    my_match_histogram(x, x) is (almost) x.
    """
    # TODO: normalised CDF of each, then searchsorted to invert the reference CDF
    raise NotImplementedError

In [ ]:
check_match_histogram(my_match_histogram)

In [ ]:
# ---- The real OpenCV route ---------------------------------------------------
# OpenCV has no histogram-matching function. You build the LUT and call cv2.LUT.
# scikit-image IS installed in this env, so you can check yourself against it:
#   from skimage.exposure import match_histograms
#   ref_out = match_histograms(dark, lena)
import cv2, numpy as np
cs = my_calcHist(dark).cumsum() / dark.size
cr = my_calcHist(lena).cumsum() / lena.size
lut = np.searchsorted(cr, cs).clip(0, 255).astype(np.uint8)
assert np.array_equal(cv2.LUT(dark, lut), my_match_histogram(dark, lena))
print("match. Comparing two histograms is built in, though:")
h1 = cv2.calcHist([dark], [0], None, [256], [0,256]); cv2.normalize(h1, h1, 0, 1, cv2.NORM_MINMAX)
h2 = cv2.calcHist([lena], [0], None, [256], [0,256]); cv2.normalize(h2, h2, 0, 1, cv2.NORM_MINMAX)
for name, m in [("CORREL", cv2.HISTCMP_CORREL), ("CHISQR", cv2.HISTCMP_CHISQR),
                ("INTERSECT", cv2.HISTCMP_INTERSECT), ("BHATTACHARYYA", cv2.HISTCMP_BHATTACHARYYA)]:
    print(f"  cv2.compareHist(..., cv2.HISTCMP_{name:13}) = {cv2.compareHist(h1, h2, m):8.4f}")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Two normalised CDFs: `cs = histogram(src).cumsum() / src.size` and the same for `ref`.

Then the whole inversion is one call: `lut = np.searchsorted(cr, cs)`. For each of the 256
source CDF values it finds where that value would slot into the (sorted, increasing)
reference CDF — which is exactly G⁻¹. Clip to 0..255 and cast to uint8 before indexing.

Normalise by pixel count on **both** sides — the two images can be different sizes, and
raw counts would not be comparable.

</details>

In [ ]:
m = my_match_histogram(dark, lena)
fig, ax = plt.subplots(2, 3, figsize=(13, 7))
for col, (im, name) in enumerate([(dark,"source (board)"), (lena,"reference (lena)"), (m,"source matched to reference")]):
    ax[0,col].imshow(im, cmap="gray", vmin=0, vmax=255); ax[0,col].set_title(name); ax[0,col].axis("off")
    ax[1,col].bar(np.arange(256), my_calcHist(im), width=1); ax[1,col].set_xlim(0,255)
plt.tight_layout(); plt.show()

for name, im in [("source", dark), ("matched", m)]:
    print(f"max CDF gap vs reference, {name:8}: "
          f"{np.abs(my_calcHist(im).cumsum()/im.size - my_calcHist(lena).cumsum()/lena.size).max():.4f}")

This is how you make a batch of photos from different cameras look consistent, and how you
normalise medical scans from different machines before feeding them to a model.

In [ ]:
# Specification also subsumes equalisation: match to a flat my_calcHist.
flat = np.tile(np.arange(256, dtype=np.uint8), (64, 1))                      # every level equally likely
_imgs, _titles = [dark, my_match_histogram(dark, flat), my_equalizeHist(dark)], ["input","matched to a FLAT my_calcHist","cv2-exact equalisation"]
fig, axes = plt.subplots(1, 3, figsize=(4 * 3, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("mean abs difference between the two right-hand images:",
      np.abs(my_match_histogram(dark, flat).astype(int) - my_equalizeHist(dark).astype(int)).mean().round(2))

## Local (adaptive) histogram processing

Everything so far used **one** histogram for the whole image, so a single dark corner cannot be
fixed without wrecking the rest. Computing the histogram in a **local window** fixes that and
introduces a new problem: noise amplification in flat tiles.

**CLAHE** (Contrast Limited Adaptive Histogram Equalisation) solves both: equalise per tile,
but *clip* each tile's histogram at a ceiling and redistribute the excess, which caps how much
contrast any flat region can gain. Then bilinearly interpolate between tiles to avoid visible seams.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
_imgs, _titles = [dark, my_equalizeHist(dark), clahe.apply(dark), cv2.createCLAHE(clipLimit=40.0, tileGridSize=(8,8)).apply(dark)], ["input","global equalisation","CLAHE clip=2","CLAHE clip=40 (limit removed)"]
fig, axes = plt.subplots(1, 4, figsize=(3.4 * 4, 3.4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

### Histogram statistics for local enhancement

You do not have to equalise at all — the local **mean** and **variance** are themselves useful.
The Gonzalez rule: boost a pixel where the neighbourhood is *darker than average* and
*low-contrast*, leave everything else alone. Cheap, and it targets exactly the shadow detail.

In [ ]:
f = dark.astype(np.float64)
k = 15
lm  = cv2.blur(f, (k,k))                                   # local mean
lv  = cv2.blur(f*f, (k,k)) - lm**2                         # local variance
ls  = np.sqrt(np.maximum(lv, 0))
G, sG = f.mean(), f.std()
boost = np.where((lm <= 0.55*G) & (ls >= 0.02*sG) & (ls <= 0.45*sG), 3.5*f, f)
_imgs, _titles = [dark, np.clip(boost,0,255).astype(np.uint8), (lm/lm.max()*255).astype(np.uint8), (ls/ls.max()*255).astype(np.uint8)], ["input","local-statistics boost","local mean","local std dev"]
fig, axes = plt.subplots(1, 4, figsize=(3.4 * 4, 3.4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

---
## Think about it

1. The equalised histogram has gaps and spikes rather than being flat. Explain why a LUT can
   never split one input level into two output levels, and what that implies about the best
   possible flatness.
2. Equalise `sudoku.png` and look at the paper. The noise got worse. Explain it in terms of which
   part of the histogram owned the most pixels.
3. `my_equalizeHist(equalize(x))` is nearly `my_equalizeHist(x)` — the grader checks this. Is equalisation
   *exactly* idempotent? Find the largest discrepancy over the four demo images and explain
   where it comes from.
4. Histogram matching is monotonic, so it cannot reorder brightnesses. Give a pair of images
   where that is a fundamental limitation — where no LUT can make one look like the other.
5. CLAHE has two knobs, `clipLimit` and `tileGridSize`. Predict the failure mode of each extreme
   (tiny tiles, huge tiles, clip 1, clip 100), then verify. Which extreme reduces to plain
   global equalisation?
6. A histogram discards all spatial information. Construct two *visually completely different*
   images with **identical** histograms. (Easy once you see it — permute the pixels.) What does
   that say about using histograms for image retrieval?

In [ ]:
# Scratch space.

---
## Stretch

1. **Colour, done properly.** Equalise a colour image three ways: per-channel RGB, the V channel
   of HSV, and the L channel of Lab. Measure the hue shift each one causes (you wrote
   `my_cvtColor_HSI` in notebook 07). Rank them and explain the ranking.
2. **Your own CLAHE.** Tile the image 8×8, clip each tile's histogram at `clipLimit * mean_count`,
   redistribute the clipped excess uniformly, equalise per tile, and bilinearly interpolate the
   four surrounding tile LUTs for each pixel. Compare against `cv2.createCLAHE`. This is a real
   exercise — expect a couple of hours, and you will understand adaptive methods afterwards.
3. **Otsu from the histogram.** Now that you have `my_calcHist`, implement Otsu's threshold as a
   vectorised maximisation of between-class variance over all 256 candidates — no pixel loops.
   Compare with `cv2.THRESH_OTSU`. (This was a stretch goal in notebook 01; you now have the tool.)
4. **Histogram-based retrieval.** Compute a 3×8×8×8 RGB histogram for all ten images in `images/`,
   normalise, and rank them by chi-squared distance from `fruits.jpg`. That is CBIR, which is on
   your Computer Vision syllabus, in about 15 lines.
5. **Bi-histogram equalisation (BBHE).** Split the histogram at the mean, equalise the two halves
   independently into their own output ranges. This preserves mean brightness, which plain
   equalisation does not. Compare on `lena.jpg`, where global equalisation overbrightens.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - histograms

```python
hist = cv2.calcHist(images, channels, mask, histSize, ranges)
```
**Every argument is a list**, even for a single grayscale image:
`cv2.calcHist([img], [0], None, [256], [0, 256])`. The result is `(256, 1) float32` —
`.ravel()` it. `mask` is a uint8 0/255 array to histogram only part of the image.
A 3-D colour histogram is `cv2.calcHist([img], [0,1,2], None, [8,8,8], [0,256]*3)`.

```python
dst = cv2.equalizeHist(src)                    # uint8 single channel, no parameters
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
dst = clahe.apply(src)
```

```python
d = cv2.compareHist(H1, H2, method)
```
`HISTCMP_CORREL` (1 = identical), `HISTCMP_CHISQR` (0 = identical),
`HISTCMP_INTERSECT` (higher = closer), `HISTCMP_BHATTACHARYYA` (0 = identical).
Normalise both histograms first: `cv2.normalize(h, h, 0, 1, cv2.NORM_MINMAX)`.

```python
back = cv2.calcBackProject([img], [0,1], roi_hist, [0,180,0,256], scale=1)
```
Histogram back-projection — "where in this image do colours like my sample appear".
The basis of MeanShift/CamShift tracking, which you meet in notebook 21.

No OpenCV function does histogram **matching**; build a LUT from two CDFs and use `cv2.LUT`.